In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Point this at wherever Module 3's CSVs actually live.
# If this notebook sits in the SAME folder as the CSVs, "." is enough.
# If Module 3 saved them elsewhere, put the full path here instead.
DATA_DIR = Path(".")   # <-- edit if your CSVs live somewhere else

datasets = {
    "11 nodes": DATA_DIR / "dataset_small__(11_nodes).csv",
    "25 nodes": DATA_DIR / "dataset_medium_(25_nodes).csv",
    "50 nodes": DATA_DIR / "dataset_large__(50_nodes).csv",
}

# Sanity check before running anything else
for label, path in datasets.items():
    print(label, "→", path, "exists:", path.exists())

11 nodes → dataset_small__(11_nodes).csv exists: True
25 nodes → dataset_medium_(25_nodes).csv exists: True
50 nodes → dataset_large__(50_nodes).csv exists: True


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


FEATURE_COLS = ["distance_km", "time_of_day", "traffic_factor",
                 "road_type", "vehicle_load_pct", "weather_condition"]
TARGET_COL = "travel_time_mins"
CATEGORICAL = ["road_type", "weather_condition"]
CONTINUOUS = ["distance_km", "time_of_day", "traffic_factor", "vehicle_load_pct"]


def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


# ---------------------------------------------------------------------
# #33 - Fair Linear Regression comparison with OneHotEncoder
# ---------------------------------------------------------------------

def run_fair_lr_comparison(df, label=""):
    """
    Trains two versions of Linear Regression:
      (a) "LR-raw"    : original setup, categoricals as raw integers
      (b) "LR-onehot" : categoricals one-hot encoded via ColumnTransformer
    Also retrains RF/XGB unchanged (tree models are insensitive to this
    encoding choice) for a like-for-like comparison table.
    """
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = {}

    # (a) LR-raw: matches your original Module 4 setup
    lr_raw = LinearRegression()
    lr_raw.fit(X_train, y_train)
    results["LR-raw"] = evaluate(y_test, lr_raw.predict(X_test))

    # (b) LR-onehot: fair encoding via ColumnTransformer + Pipeline
    preprocessor = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), CONTINUOUS),
    ])
    lr_onehot = Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression()),
    ])
    lr_onehot.fit(X_train, y_train)
    results["LR-onehot"] = evaluate(y_test, lr_onehot.predict(X_test))

    # RF/XGB unchanged, for reference in the same table
    rf = RandomForestRegressor(random_state=42, n_estimators=200)
    rf.fit(X_train, y_train)
    results["RF"] = evaluate(y_test, rf.predict(X_test))

    xgb = XGBRegressor(random_state=42, n_estimators=200)
    xgb.fit(X_train, y_train)
    results["XGB"] = evaluate(y_test, xgb.predict(X_test))

    print(f"\n=== Fair LR comparison ({label}) ===")
    for name, m in results.items():
        print(f"  {name:10s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  R2={m['R2']:.4f}")

    return results


# ---------------------------------------------------------------------
# #35 - Ablation: time_of_day vs traffic_factor redundancy
# ---------------------------------------------------------------------

def run_ablation(df, label="", model_name="XGB"):
    """
    Trains three variants of the same model (default XGBoost, since it's
    your primary model) on different feature subsets:
      - "time_only"    : distance, road_type, load, weather, TIME_OF_DAY (no traffic_factor)
      - "traffic_only" : distance, road_type, load, weather, TRAFFIC_FACTOR (no time_of_day)
      - "both"          : all six features (your current setup)
    Shows how much predictive power time_of_day loses once traffic_factor
    (which is generated FROM hour in Module 3) is available, and vice versa.
    """
    base_cols = ["distance_km", "road_type", "vehicle_load_pct", "weather_condition"]
    variants = {
        "time_only":    base_cols + ["time_of_day"],
        "traffic_only": base_cols + ["traffic_factor"],
        "both":         base_cols + ["time_of_day", "traffic_factor"],
    }

    y = df[TARGET_COL]
    results = {}

    for variant_name, cols in variants.items():
        X = df[cols]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        model = XGBRegressor(random_state=42, n_estimators=200)
        model.fit(X_train, y_train)
        results[variant_name] = evaluate(y_test, model.predict(X_test))

    print(f"\n=== Ablation: time_of_day vs traffic_factor ({label}, {model_name}) ===")
    for name, m in results.items():
        print(f"  {name:14s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  R2={m['R2']:.4f}")

    return results


if __name__ == "__main__":
    # ---- EDIT THESE PATHS to your actual Module 3 output CSVs ----
    datasets = {
        "11 nodes": "dataset_small__(11_nodes).csv",
        "25 nodes": "dataset_medium_(25_nodes).csv",
        "50 nodes": "dataset_large__(50_nodes).csv",
    }

    all_fairness_results = {}
    all_ablation_results = {}

    for label, path in datasets.items():
        df = pd.read_csv(path)
        all_fairness_results[label] = run_fair_lr_comparison(df, label=label)
        all_ablation_results[label] = run_ablation(df, label=label)

    print("\n\nCopy the printed tables above back into the conversation.")


for label, path in datasets.items():
    df = pd.read_csv(path)
    run_fair_lr_comparison(df, label=label)
    run_ablation(df, label=label)


=== Fair LR comparison (11 nodes) ===
  LR-raw      MAE=12.6062  RMSE=17.2462  R2=0.8120
  LR-onehot   MAE=11.5041  RMSE=15.9025  R2=0.8401
  RF          MAE=6.1444  RMSE=9.7144  R2=0.9403
  XGB         MAE=6.5639  RMSE=10.0508  R2=0.9361

=== Ablation: time_of_day vs traffic_factor (11 nodes, XGB) ===
  time_only       MAE=6.7699  RMSE=10.4849  R2=0.9305
  traffic_only    MAE=6.8332  RMSE=10.5479  R2=0.9297
  both            MAE=6.5220  RMSE=9.9914  R2=0.9369

=== Fair LR comparison (25 nodes) ===
  LR-raw      MAE=12.8768  RMSE=17.7606  R2=0.7938
  LR-onehot   MAE=12.2175  RMSE=16.7645  R2=0.8163
  RF          MAE=5.8434  RMSE=9.4296  R2=0.9419
  XGB         MAE=6.1060  RMSE=9.3883  R2=0.9424

=== Ablation: time_of_day vs traffic_factor (25 nodes, XGB) ===
  time_only       MAE=6.1058  RMSE=9.5222  R2=0.9407
  traffic_only    MAE=6.5511  RMSE=10.1387  R2=0.9328
  both            MAE=6.0822  RMSE=9.3662  R2=0.9427

=== Fair LR comparison (50 nodes) ===
  LR-raw      MAE=12.6413  RMSE

In [10]:
"""
exp_baseline_and_gridsearch.py

Addresses:
  #43 - No simple physics/heuristic baseline for time prediction. Adds a
        fixed-average-speed baseline (no ML, no traffic/weather/load
        adjustment) so you can show ML's actual value-add over the
        simplest possible estimator.

  #36 - GridSearchCV is mentioned but undocumented. Runs an explicit,
        printed grid search for RF and XGB with a defined param_grid,
        reporting best_params_ and the top cross-validation results --
        this is what goes into your thesis table/appendix.

Run once per dataset size.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


FEATURE_COLS = ["distance_km", "time_of_day", "traffic_factor",
                 "road_type", "vehicle_load_pct", "weather_condition"]
TARGET_COL = "travel_time_mins"

# Approximate mean speed across your three road types (km/min), used only
# for the baseline -- NOT the same as any per-road-type speed used in
# Module 3's actual generator.
BASELINE_AVG_SPEED_KM_PER_MIN = 0.85  # ~51 km/h, a plausible fleet-wide average


def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def physics_baseline_predict(distance_km):
    """distance / fixed average speed -- the simplest possible estimator,
    with no traffic, weather, load, or road-type adjustment at all."""
    return distance_km / BASELINE_AVG_SPEED_KM_PER_MIN


def run_baseline_comparison(df, label=""):
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    baseline_pred = physics_baseline_predict(X_test["distance_km"].values)
    results = {"Physics baseline (fixed speed)": evaluate(y_test, baseline_pred)}

    print(f"\n=== Physics baseline vs ML ({label}) ===")
    print(f"  {'Physics baseline':22s}  "
          f"MAE={results['Physics baseline (fixed speed)']['MAE']:.4f}  "
          f"RMSE={results['Physics baseline (fixed speed)']['RMSE']:.4f}  "
          f"R2={results['Physics baseline (fixed speed)']['R2']:.4f}")
    print("  (Compare this row against your existing LR/RF/XGB rows in Table 5.1)")

    return results


def run_documented_gridsearch(df, label=""):
    """
    Explicit, printed grid search for RF and XGB. Prints best_params_ and
    the top-5 parameter combinations by mean CV score, so you have a
    concrete table/appendix entry instead of an undocumented one-liner.
    """
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    rf_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10],
    }
    xgb_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 6, 9],
        "learning_rate": [0.05, 0.1, 0.2],
    }

    print(f"\n=== GridSearchCV: Random Forest ({label}) ===")
    rf_search = GridSearchCV(
        RandomForestRegressor(random_state=42), rf_grid,
        cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
    )
    rf_search.fit(X_train, y_train)
    print(f"  Best params: {rf_search.best_params_}")
    print(f"  Best CV MAE: {-rf_search.best_score_:.4f}")
    test_pred = rf_search.predict(X_test)
    print(f"  Test set   : {evaluate(y_test, test_pred)}")

    print(f"\n=== GridSearchCV: XGBoost ({label}) ===")
    xgb_search = GridSearchCV(
        XGBRegressor(random_state=42), xgb_grid,
        cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
    )
    xgb_search.fit(X_train, y_train)
    print(f"  Best params: {xgb_search.best_params_}")
    print(f"  Best CV MAE: {-xgb_search.best_score_:.4f}")
    test_pred = xgb_search.predict(X_test)
    print(f"  Test set   : {evaluate(y_test, test_pred)}")

    return {
        "rf_best_params": rf_search.best_params_,
        "rf_best_cv_mae": -rf_search.best_score_,
        "xgb_best_params": xgb_search.best_params_,
        "xgb_best_cv_mae": -xgb_search.best_score_,
    }


if __name__ == "__main__":
    datasets = {
        "11 nodes": "dataset_small__(11_nodes).csv",
        "25 nodes": "dataset_medium_(25_nodes).csv",
        "50 nodes": "dataset_large__(50_nodes).csv",
    }

    for label, path in datasets.items():
        df = pd.read_csv(path)
        run_baseline_comparison(df, label=label)
        run_documented_gridsearch(df, label=label)  # slower -- comment out if time-constrained

    print("\n\nCopy the printed tables above back into the conversation.")

for label, path in datasets.items():
    df = pd.read_csv(path)
    run_baseline_comparison(df, label=label)
    run_documented_gridsearch(df, label=label)  # slower — grid search


=== Physics baseline vs ML (11 nodes) ===
  Physics baseline        MAE=26.8394  RMSE=40.6874  R2=-0.0467
  (Compare this row against your existing LR/RF/XGB rows in Table 5.1)

=== GridSearchCV: Random Forest (11 nodes) ===
  Best params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 300}
  Best CV MAE: 5.8949
  Test set   : {'MAE': 6.088004824906665, 'RMSE': np.float64(9.615781572768805), 'R2': 0.9415407156188494}

=== GridSearchCV: XGBoost (11 nodes) ===
  Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}
  Best CV MAE: 5.7046
  Test set   : {'MAE': 5.856859484116236, 'RMSE': np.float64(9.410271412572282), 'R2': 0.9440128172163746}

=== Physics baseline vs ML (25 nodes) ===
  Physics baseline        MAE=25.1098  RMSE=39.1813  R2=-0.0034
  (Compare this row against your existing LR/RF/XGB rows in Table 5.1)

=== GridSearchCV: Random Forest (25 nodes) ===
  Best params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 300}
  Best CV MAE: 5

In [11]:
"""
exp_ood_and_permutation_importance.py

Addresses:
  #54 - No out-of-distribution (OOD) test. Trains on non-storm data and
        tests on storm-only holdout, to see how badly performance
        degrades when deployment conditions differ from training.

  #51 - Feature importance analysis is too uncritical (built-in
        feature_importances_ can be misleading, e.g. biased toward
        high-cardinality features). Adds permutation importance as a
        complementary, model-agnostic check.

Run once per dataset size, or just on 11 nodes if time-constrained --
this is diagnostic, not a primary result table.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


FEATURE_COLS = ["distance_km", "time_of_day", "traffic_factor",
                 "road_type", "vehicle_load_pct", "weather_condition"]
TARGET_COL = "travel_time_mins"


def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


# ---------------------------------------------------------------------
# #54 - Out-of-distribution test
# ---------------------------------------------------------------------

def run_ood_weather_shift(df, label=""):
    """
    Trains on everything EXCEPT storm conditions (weather_condition != 3),
    tests on storm-only holdout (weather_condition == 3). This directly
    measures the distribution-shift risk your Limitations section already
    names qualitatively.
    """
    train_df = df[df["weather_condition"] != 3]
    test_df = df[df["weather_condition"] == 3]

    if len(test_df) < 20:
        print(f"  [{label}] Too few storm records ({len(test_df)}) for a "
              f"reliable OOD test -- consider regenerating with more data "
              f"or a larger storm probability.")
        return None

    X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
    X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET_COL]

    # Also build an in-distribution comparison: same train set, tested on
    # a random (non-storm) held-out split, for a fair before/after picture.
    X_train_sub, X_val_indist, y_train_sub, y_val_indist = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )

    results = {}
    for name, model in [("RF", RandomForestRegressor(random_state=42, n_estimators=200)),
                         ("XGB", XGBRegressor(random_state=42, n_estimators=200))]:
        model.fit(X_train_sub, y_train_sub)
        results[f"{name}_in_distribution"] = evaluate(y_val_indist, model.predict(X_val_indist))
        results[f"{name}_ood_storm"] = evaluate(y_test, model.predict(X_test))

    print(f"\n=== OOD test: trained without storm, tested on storm-only ({label}) ===")
    for name, m in results.items():
        print(f"  {name:20s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  R2={m['R2']:.4f}")
    print("  Compare *_in_distribution vs *_ood_storm rows: a large MAE/RMSE "
          "jump indicates the model does not generalize to unseen weather severity.")

    return results


# ---------------------------------------------------------------------
# #51 - Permutation importance (complements built-in feature_importances_)
# ---------------------------------------------------------------------

def run_permutation_importance(df, label="", model_name="XGB"):
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    if model_name == "XGB":
        model = XGBRegressor(random_state=42, n_estimators=200)
    else:
        model = RandomForestRegressor(random_state=42, n_estimators=200)
    model.fit(X_train, y_train)

    builtin_importance = dict(zip(FEATURE_COLS, model.feature_importances_))

    perm_result = permutation_importance(
        model, X_test, y_test, n_repeats=20, random_state=42, scoring="neg_mean_absolute_error"
    )
    perm_importance = dict(zip(FEATURE_COLS, perm_result.importances_mean))
    perm_std = dict(zip(FEATURE_COLS, perm_result.importances_std))

    print(f"\n=== Permutation importance vs built-in ({label}, {model_name}) ===")
    print(f"  {'Feature':20s}  {'Built-in':>10s}  {'Permutation':>14s}  {'(+/- std)':>10s}")
    for feat in FEATURE_COLS:
        print(f"  {feat:20s}  {builtin_importance[feat]:>10.4f}  "
              f"{perm_importance[feat]:>14.4f}  {perm_std[feat]:>10.4f}")
    print("  Large disagreement between the two columns (e.g. a feature ranked "
          "high by built-in importance but low by permutation importance) "
          "signals the built-in ranking may be biased for that feature.")

    return {"builtin": builtin_importance, "permutation": perm_importance, "perm_std": perm_std}


if __name__ == "__main__":
    datasets = {
        "11 nodes": "dataset_small__(11_nodes).csv",
        "25 nodes": "dataset_medium_(25_nodes).csv",
        "50 nodes": "dataset_large__(50_nodes).csv",
    }

    for label, path in datasets.items():
        df = pd.read_csv(path)
        run_ood_weather_shift(df, label=label)
        run_permutation_importance(df, label=label, model_name="XGB")

    print("\n\nCopy the printed tables above back into the conversation.")


=== OOD test: trained without storm, tested on storm-only (11 nodes) ===
  RF_in_distribution    MAE=5.5769  RMSE=8.0378  R2=0.9545
  RF_ood_storm          MAE=11.0565  RMSE=15.5140  R2=0.8690
  XGB_in_distribution   MAE=6.0418  RMSE=8.6974  R2=0.9467
  XGB_ood_storm         MAE=10.6346  RMSE=14.9655  R2=0.8781
  Compare *_in_distribution vs *_ood_storm rows: a large MAE/RMSE jump indicates the model does not generalize to unseen weather severity.

=== Permutation importance vs built-in (11 nodes, XGB) ===
  Feature                 Built-in     Permutation   (+/- std)
  distance_km               0.1941         19.4811      0.6521
  time_of_day               0.0097          0.6761      0.1056
  traffic_factor            0.0703          7.6226      0.6253
  road_type                 0.6314         22.0876      0.9522
  vehicle_load_pct          0.0074          0.5038      0.1607
  weather_condition         0.0871          3.6552      0.2449
  Large disagreement between the two columns (

In [12]:
"""
exp_multiseed_stability.py

Addresses:
  #44 / #55 - All of Table 5.1's MAE/RMSE/R2 figures come from a single
              seed (42). The ground-truth regret evaluation you already
              ran covers seed variance for Phase 3 routing (5 seeds), but
              Phase 2 model performance itself is still single-seed.
              This runs RF and XGB training across 5 different
              train/test splits (seeds 42, 43, 44, 45, 46) and reports
              mean +/- std, so Table 5.1 can carry error bars instead of
              point estimates.

Run once per dataset size. This is the fast part (no VRPTW solving) so
it's cheap to run on all three sizes even if you skip other experiments.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


FEATURE_COLS = ["distance_km", "time_of_day", "traffic_factor",
                 "road_type", "vehicle_load_pct", "weather_condition"]
TARGET_COL = "travel_time_mins"
SEEDS = [42, 43, 44, 45, 46]


def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def run_multiseed(df, label=""):
    rows = []
    for seed in SEEDS:
        X = df[FEATURE_COLS]
        y = df[TARGET_COL]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed
        )

        rf = RandomForestRegressor(random_state=seed, n_estimators=200)
        rf.fit(X_train, y_train)
        rf_metrics = evaluate(y_test, rf.predict(X_test))

        xgb = XGBRegressor(random_state=seed, n_estimators=200)
        xgb.fit(X_train, y_train)
        xgb_metrics = evaluate(y_test, xgb.predict(X_test))

        rows.append({
            "seed": seed,
            "RF_MAE": rf_metrics["MAE"], "RF_RMSE": rf_metrics["RMSE"], "RF_R2": rf_metrics["R2"],
            "XGB_MAE": xgb_metrics["MAE"], "XGB_RMSE": xgb_metrics["RMSE"], "XGB_R2": xgb_metrics["R2"],
        })

    df_results = pd.DataFrame(rows)
    summary = df_results.drop(columns="seed").agg(["mean", "std"])

    print(f"\n=== Multi-seed stability ({label}) ===")
    print(df_results.to_string(index=False))
    print("\nMean +/- std across 5 seeds:")
    print(summary)

    fname = f"multiseed_stability_{label.replace(' ', '')}.csv"
    df_results.to_csv(fname, index=False)
    print(f"Saved: {fname}")

    return df_results, summary


if __name__ == "__main__":
    datasets = {
        "11 nodes": "dataset_small__(11_nodes).csv",
        "25 nodes": "dataset_medium_(25_nodes).csv",
        "50 nodes": "dataset_large__(50_nodes).csv",
    }

    for label, path in datasets.items():
        df = pd.read_csv(path)
        run_multiseed(df, label=label)

    print("\n\nCopy the printed tables above back into the conversation.")


=== Multi-seed stability (11 nodes) ===
 seed   RF_MAE  RF_RMSE    RF_R2  XGB_MAE  XGB_RMSE   XGB_R2
   42 6.144421 9.714450 0.940335 6.563913 10.050771 0.936132
   43 6.000152 9.764841 0.938094 6.387933  9.908456 0.936260
   44 5.718170 8.751762 0.946491 6.192056  9.476813 0.937257
   45 5.542492 8.976185 0.946540 6.294974  9.825710 0.935942
   46 5.735021 8.725844 0.949664 6.407582  9.476291 0.940634

Mean +/- std across 5 seeds:
        RF_MAE   RF_RMSE     RF_R2   XGB_MAE  XGB_RMSE    XGB_R2
mean  5.828051  9.186616  0.944225  6.369292  9.747608  0.937245
std   0.240822  0.514452  0.004817  0.138407  0.260203  0.001961
Saved: multiseed_stability_11nodes.csv

=== Multi-seed stability (25 nodes) ===
 seed   RF_MAE  RF_RMSE    RF_R2  XGB_MAE  XGB_RMSE   XGB_R2
   42 5.843437 9.429621 0.941881 6.105979  9.388259 0.942390
   43 5.497438 8.280371 0.949196 6.004334  8.730121 0.943527
   44 5.688249 9.076471 0.944747 6.032070  9.419722 0.940489
   45 5.377102 8.448791 0.951423 5.832753  8